# Install Packages and Setup Variables


In [ ]:
!pip install -q openai==2.8.1 google-genai==1.51.0

In [ ]:
import os

# Set the "OPENAI_API_KEY" and "GOOGLE_API_KEY" in the Python environment. Will be used by OpenAI client later.

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    pass  # Running locally — keys are expected in the environment

In [ ]:
# False: Generate the embedding for the dataset. (Associated cost with using OpenAI endpoint)
# True: Load the dataset that already has the embedding vectors.
load_embedding = False

# Load Dataset


## Download Dataset (JSON)


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

## Read File


In [ ]:
# Split the input text into chunks of specified size.
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])

    return chunks

In [ ]:
import csv

chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        chunks.extend(split_into_chunks(row[1]))

In [ ]:
print("number of articles:", idx)
print("number of chunks:", len(chunks))

In [ ]:
import pandas as pd

# Convert the JSON list to a Pandas Dataframe
df = pd.DataFrame(chunks, columns=["chunk"])

df.keys()

# Generate Embedding


In [ ]:
from openai import OpenAI

client = OpenAI()


# Defining a function that converts a text to embedding vector using OpenAI's Ada model.
def get_embedding(text):
    try:
        # Remove newlines
        text = text.replace("\n", " ")
        res = client.embeddings.create(input=[text], model="text-embedding-3-small")

        return res.data[0].embedding

    except:
        return None

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

# Generate embedding
if not load_embedding:
    print("Generating embeddings...")
    embeddings = []
    for index, row in tqdm(df.iterrows()):
        # df.at[index, 'embedding'] = get_embedding( row['chunk'] )
        embeddings.append(get_embedding(row["chunk"]))

    embeddings_values = pd.Series(embeddings)
    df.insert(loc=1, column="embedding", value=embeddings_values)

# Or, load the embedding from the file.
else:
    print("Loaded the embedding file.")
    # Load the file as a CSV
    df = pd.read_csv("mini-llama-articles-with_embeddings.csv")
    # Convert embedding column to an array
    df["embedding"] = df["embedding"].apply(lambda x: np.array(eval(x)), 0)

In [ ]:
# df.to_csv('mini-llama-articles-with_embeddings.csv')

# User Question


In [ ]:
# Define the user question, and convert it to embedding.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = get_embedding(QUESTION)

len(QUESTION_emb)

# Test Cosine Similarity


Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [ ]:
BAD_SOURCE_emb = get_embedding("The sky is blue.")
GOOD_SOURCE_emb = get_embedding("LLaMA2 model has a total of 2B parameters.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# A sample that how a good piece of text can achieve high similarity score compared
# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

# Calculate Cosine Similarities


In [ ]:
# The similarity between the questions and each part of the essay.
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

In [ ]:
import numpy as np

number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

In [ ]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

# Augment the Prompt


In [ ]:
from google import genai
from google.genai import types
from google.genai.types import HttpOptions

# Initialize client
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # System instructions for the AI assistant
    system_instruction = (
        "You are an assistant and expert in answering questions from a chunks of content. "
        "Only answer AI-related question, else say that you cannot answer this question."
    )

    # Create a user prompt with the user's question
    prompt = (
        "Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        "Please provide an informative and accurate answer to the following question based on the available context. Be concise and take your time. \nQuestion: {}\nAnswer:"
    )

    # Add the retrieved pieces of text to the prompt
    formatted_prompt = prompt.format("".join(df.chunk[indices]), QUESTION)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=formatted_prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            system_instruction=system_instruction,
            temperature=0.0,
        )
    )

    res = response.text

except Exception as e:
    print(f"An error occurred: {e}")

print(res)

## How Augmenting the Prompt can address knowledge cutoff limitations and hallucinations

In [ ]:
# Consider this as a retrieved chunk
# https://ai.meta.com/blog/llama-4-multimodal-intelligence (Summarized Content)

Example_chunk ="""
Meta has unveiled the **Llama 4 herd**, a new generation of open-weight, natively multimodal AI models designed to push the boundaries of performance, efficiency, and accessibility. The release includes **Llama 4 Scout** and **Llama 4 Maverick**, alongside a preview of the massive **Llama 4 Behemoth** teacher model. These models introduce advanced **mixture-of-experts (MoE) architectures**, native text–image integration, and record-breaking context lengths, establishing Llama 4 as a major leap in multimodal AI innovation.
**Llama 4 Scout** is a **17B active parameter model with 16 experts** (109B total parameters) that offers **10 million token context length**, far exceeding Llama 3’s 128K limit. Scout’s architecture leverages **interleaved attention layers (iRoPE)** and inference-time temperature scaling to achieve superior length generalization, enabling complex tasks like multi-document summarization, codebase reasoning, and long-context retrieval. Despite its smaller size, Scout surpasses prior Llama models and competitors such as Gemma 3 and Gemini 2.0 Flash-Lite in performance, all while being deployable on a **single NVIDIA H100 GPU**. Its multimodal training allows strong image grounding, multi-image reasoning, and visual question answering.
**Llama 4 Maverick**, also with **17B active parameters**, uses **128 experts** and totals **400B parameters**, alternating dense and MoE layers for inference efficiency. It rivals or exceeds larger models like **GPT-4o, Gemini 2.0 Flash, and DeepSeek v3** on benchmarks for coding, reasoning, multilinguality, and multimodal tasks. Its efficient design makes it deployable on a single **NVIDIA H100 DGX host**, balancing performance with cost-effectiveness. Maverick was refined through a revamped **post-training pipeline**—lightweight supervised fine-tuning, continuous **online reinforcement learning (RL)** with adaptive difficulty filtering, and direct preference optimization—resulting in superior reasoning, conversational fluency, and multimodal understanding. Maverick’s **chat version achieved an ELO score of 1417 on LMArena**, reflecting best-in-class general assistant capabilities.
At the top of the hierarchy, **Llama 4 Behemoth** serves as the **teacher model**, with **288B active parameters, 16 experts, and nearly 2 trillion total parameters**. It outperforms **GPT-4.5, Claude Sonnet 3.7, and Gemini 2.0 Pro** on STEM benchmarks like MATH-500 and GPQA Diamond. Behemoth’s scale demanded innovative training strategies, including pruning 95% of supervised fine-tuning data, advanced reinforcement learning with dynamically filtered prompts, and an optimized asynchronous MoE-parallelized RL infrastructure. These techniques boosted reasoning, coding, and multilingual capabilities while maintaining instruction-following reliability.
All Llama 4 models are **natively multimodal**, trained with large-scale text, image, and video data, and leverage **Meta’s MetaP technique** for reliable hyperparameter scaling. They support **200 languages**, with 10x more multilingual tokens than Llama 3, and employ **FP8 precision** for efficient training across trillions of tokens. Safety remains a priority: Meta integrates **Llama Guard**, **Prompt Guard**, and **CyberSecEval** for content protection, while **Generative Offensive Agent Testing (GOAT)** automates adversarial red-teaming. Llama 4 also significantly reduces **political and social bias**, refusing fewer prompts and responding more neutrally than Llama 3.
In sum, **Llama 4 Scout, Maverick, and Behemoth** represent a major leap in open AI research: compact yet powerful models with unmatched context length, multimodal fluency, reasoning power, and efficiency. By making Scout and Maverick openly available on **llama.com and Hugging Face**, Meta empowers developers, enterprises, and researchers worldwide to build the next generation of AI-driven experiences."""

In [ ]:
QUESTION = "How many parameters does LLaMA 4 model have?"

system_instruction = (
    "You are an assistant and expert in answering questions from a chunks of content. "
    "Only answer AI-related question, else say that you cannot answer this question."
)

# Combining the context with the user's question
prompt = (
    "Read the following informations that might contain the context you require to answer the question. You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based on the available context. Be concise and take your time. \nQuestion: {}\nAnswer:"
)

formatted_prompt = prompt.format(Example_chunk, QUESTION)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

res = response.text
print(res)

# Without Augmentation


Test the Gemini API to answer the same question without the addition of retrieved documents. Basically, the LLM will use its knowledge to answer the question.


In [ ]:
# Test without retrieved documents
QUESTION = "How many parameters does LLaMA 4 model have?"

# System instructions
system_instruction = "You are an assistant and expert in answering questions."

# Simple prompt without context
prompt = "Be concise and take your time to answer the following question. \nQuestion: {}\nAnswer:"
formatted_prompt = prompt.format(QUESTION)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction=system_instruction,
        temperature=0.0,
    )
)

res = response.text
print(res)

## Optional: Similarity Score Bar Chart

Visualize how cosine similarity scores compare across the top retrieved chunks
so you can quickly see which chunks are the most relevant to your question.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for script execution
import matplotlib.pyplot as plt

# Use a dedicated OpenAI client (the main `client` was re-assigned to Gemini above)
_oai = OpenAI()

def get_embedding_optional(text):
    text = text.replace("\n", " ")
    res = _oai.embeddings.create(input=[text], model="text-embedding-3-small")
    return res.data[0].embedding

# Re-run similarity search with a fresh question
QUESTION_CHART = "How many parameters LLaMA2 model has?"
QUESTION_CHART_emb = get_embedding_optional(QUESTION_CHART)
sims = cosine_similarity([QUESTION_CHART_emb], df["embedding"].tolist())[0]
top_n = 5
top_indices = np.argsort(sims)[::-1][:top_n]
top_scores = sims[top_indices]
top_labels = [f"Chunk {i}" for i in top_indices]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(top_labels[::-1], top_scores[::-1], color="steelblue")
ax.set_xlabel("Cosine Similarity")
ax.set_title(f"Top {top_n} Chunks by Similarity\nQuestion: \"{QUESTION_CHART}\"")
ax.bar_label(bars, fmt="%.4f", padding=4)
plt.tight_layout()
plt.savefig("similarity_chart.png", dpi=100)
plt.close()
print("Bar chart saved to similarity_chart.png")
for label, score in zip(top_labels, top_scores):
    print(f"  {label}: {score:.4f}")

## Optional: RAG Quality — With vs. Without Augmentation

Compare the model's answer quality when context is provided versus when it answers
from its own knowledge alone, using the same question.

In [ ]:
COMPARE_QUESTION = "How many parameters LLaMA2 model has?"
COMPARE_QUESTION_emb = get_embedding_optional(COMPARE_QUESTION)
compare_sims = cosine_similarity([COMPARE_QUESTION_emb], df["embedding"].tolist())
top_idx = np.argsort(compare_sims[0])[::-1][:3]

# With augmentation
rag_prompt = (
    "Read the following information that might contain the context you require to answer the question. "
    "Content starts with <START_OF_CONTEXT> and ends with <END_OF_CONTEXT>.\n\n"
    "<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
    "Question: {}\nAnswer:"
)
formatted_rag = rag_prompt.format("".join(df.chunk[top_idx]), COMPARE_QUESTION)
rag_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=formatted_rag,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction="You are an expert assistant.",
        temperature=0.0,
    )
)

# Without augmentation
bare_prompt = "Answer the following question briefly.\nQuestion: {}\nAnswer:"
bare_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=bare_prompt.format(COMPARE_QUESTION),
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0),
        system_instruction="You are an expert assistant.",
        temperature=0.0,
    )
)

print("=== WITH RAG CONTEXT ===")
print(rag_response.text)
print("\n=== WITHOUT CONTEXT (model knowledge only) ===")
print(bare_response.text)